## Currency Conversion Tool

In [ ]:
from langchain_core.tools import tool
import requests
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv


load_dotenv()
api_key='Your_API_KEY_HERE'

In [5]:
#Create Tools

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """This function fetches a currency conversion factor between a base_currency and a target_currency"""
    # Logic to fetch conversion factor
    url = f'https://v6.exchangerate-api.com/v6/{api_key}/pair/{base_currency}/{target_currency}'

    response = requests.get(url)
    data = response.json()

    return data['conversion_rate']


@tool
def convert(base_currency_value: int, conversion_factor: float) -> float:
    """This function calculates the target_currency value of a given base_currency value using conversion factor"""

    #Logic to convert base_currency_value to target_currency
    target_currency_value = base_currency_value * conversion_factor

    return target_currency_value



In [6]:
conversion_factor = get_conversion_factor.invoke({
    'base_currency': 'USD',
    'target_currency': 'NPR'
})

target_value = convert.invoke({
    'base_currency_value': 121,
    'conversion_factor': 153.1049
})


print(conversion_factor)
print(f'NPR: {target_value}') 

153.1049
NPR: 18525.6929


In [7]:
llm = ChatGoogleGenerativeAI(
    model='gemini-3.5-flash'
)

In [8]:
#Tool Binding
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [9]:
messages = [HumanMessage(
    content="What is the conversion factor between NPR and USD, "
        "and convert 1530019 NPR to USD"
    )
]

In [10]:
messages

[HumanMessage(content='What is the conversion factor between NPR and USD, and convert 1530019 NPR to USD', additional_kwargs={}, response_metadata={})]

In [11]:
# Round 1: model asks for get_conversion_factor
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)



In [12]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'NPR', 'target_currency': 'USD'},
  'id': 'call_3920695',
  'type': 'tool_call'}]

In [13]:
# Execute the first tool call (get_conversion_factor)
for tool_call in ai_message.tool_calls:
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        # tool_message1.content is a plain stringified float (e.g. "153.1049"),
        # not a JSON object, so we parse it as a float directly
        conversion_rate = float(tool_message1.content)
        messages.append(tool_message1)

In [16]:
ai_message2 = llm_with_tools.invoke(messages)
messages.append(ai_message2)


In [17]:
ai_message2.tool_calls

[{'name': 'convert',
  'args': {'base_currency_value': 1530019, 'conversion_factor': 0.006531},
  'id': 'call_7190934',
  'type': 'tool_call'}]

In [24]:
#Execute the second tool call
for tool_call in ai_message2.tool_calls:
    if tool_call['name'] == 'convert':
        tool_call['args', 'conversion_rate'] = conversion_rate

        tool_message2 = convert.invoke(tool_call)

        messages.append(tool_message2)



In [25]:
messages

[HumanMessage(content='What is the conversion factor between NPR and USD, and convert 1530019 NPR to USD', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "NPR", "target_currency": "USD"}'}, '__gemini_function_call_thought_signatures__': {'call_3920695': 'EpwFCpkFARFNMg/+gitn7Md6B1qWKOWsiNzevxG50YavbjCHXS+Kavq+hWBj5fRb4UNyIKP+XlKKOQT7PLHYJJZ7Q6ukeEnZR1u+W3rHMql9dQnUBM4ikSEcG2+dcG/0vhhUWwajNcJtnp3oaZTx30Yebk2AUs1tQkOL0jBLQ8DWNAVYKJupSsMSmjyHt4E8oNQk9SrrAJ15NKVNe0BaM+H/A3faM3D8ro+36K6yPCIKZ5ZyAjUzcRB4UH+7xE50wH4xMx1ly/pvdL3N4QZWGUgoVDmVZYSTHHcZ6ZocQczqNRWzRtmKbwJTDvfChmyRP5Y3En1JgFVvvBDmBIymGVl3Jt/aEF/Bg8+RRg9Mvq6/bbnMFolK/FWmPH3nCdwGIeq4FNDAbOclf4Fiz/sbdgSoDIvHBQEDMon+xjPlfHRFcWkWBEO66ADaIqC6TRmXEP0FKQz5dT3MWiH8gWKlZalC3yzQwR5gaYUmw6CDev4ueQistkVDc0rMz2uRv4WuVrx+JqwTRkhevpzUpPTLWTzQcsTnZ4mPhWLTdo1XRCuYre6fRPbO4RZSHJuPCgbGwaoOoZvlGRvwvU4cjkMi+kzqwK+7BjTdgcvaDtbEczmcVA

In [26]:
final_response = llm_with_tools.invoke(messages)
final_response.content

[{'type': 'text',
  'text': 'The conversion factor from NPR to USD is **0.006531**.\n\nConverting **1,530,019 NPR** to USD:\n1,530,019 NPR * 0.006531 = **9,992.55 USD**',
  'extras': {'signature': 'EokCCoYCARFNMg87AHwXqf+AneiN1qDmy9kYlFYULcS2/MrY7VrGADd42ckz2OUAwUx84rMgziPNZIljTWZ/Jxmx17li5lTtE83fzSgvRRK3sszCW1l+HI/zNAyg6vpQEM6hVCg0ht4oIA+fysr2rSQDKq/Jz7q980EYbWHgunwdqVi2KFKjxB1lUsNjmY7Y+kHBhwW4ricLSpVNIoaq0rf3sMk1o52aK2WELUgSrTM5SZ710U7zTJ2uk2iLxm4sr1H8wb8z3bP+TVoMW+V8w8xdIcJ+GCdvyNEUw7wetw72gTkCndbsNi3jVR7PE2OOsLa8NeYKWEGbcFEBibyDXTqmJKZlQy/48u/nRQ=='}}]